In [1]:
import os, json, numpy as np, pandas as pd
SEED=42
P2_OUT='/kaggle/input/datasets/risha8750/phase2-latest-outputs/phase2_outputs'
P2_CACHE='/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache'
P3_CSV='/kaggle/input/datasets/risha8750/phase3-results-csv/phase3_results.csv'
OUT='/kaggle/working/phase4_decision'
for s in ['','plots','retry_outputs','meshes_v2']: os.makedirs(f'{OUT}/{s}',exist_ok=True)

In [2]:
p3=pd.read_csv(P3_CSV)
p2=pd.read_csv(f'{P2_OUT}/test_per_building.csv')
cfg=json.load(open(f'{P2_CACHE}/cache_config.json'))
print(p2.columns.tolist(), p2.shape)
print(cfg)
print(p3.building_id.is_unique, p2.building_id.is_unique if 'building_id' in p2 else 'no id col')
print(len(set(p3.building_id)&set(p2.building_id)) if 'building_id' in p2 else '')
print(p3.isna().sum()[lambda s:s>0])
conf=['corner_confidence','edge_confidence','model_confidence','external_confidence','combined_confidence']
print(p3[conf].describe().loc[['min','max']])
both_missing=~p3.osm_available & ~p3.microsoft_available
print(both_missing.sum(), np.allclose(p3.loc[both_missing,'combined_confidence'],p3.loc[both_missing,'model_confidence']))
print(p3.confidence_class.value_counts(), p3.coverage_class.value_counts())

['building_id', 'corner_tp', 'corner_pred', 'corner_true', 'corner_offset_sum', 'edge_tp', 'edge_pred', 'edge_true'] (3235, 8)
{'source_root': '/kaggle/input/notebooks/risha8750/phase-1b/building3d_tallinn', 'roof_archive': '/kaggle/input/notebooks/risha8750/phase-1b/building3d_tallinn/archives/tallinn_roof_wireframes.zip', 'building_archive': '/kaggle/input/notebooks/risha8750/phase-1b/building3d_tallinn/archives/tallinn_building_pointclouds.zip', 'manifest': '/kaggle/input/notebooks/risha8750/phase-1b/building3d_tallinn/phase1_outputs', 'point_count': 2048, 'max_corners': 64, 'max_edges': 128, 'seed': 42, 'samples': 31984, 'splits': {'train': 25579, 'test': 3235, 'val': 3170}}
True True
321
mean_corner_spread_m               7
max_corner_spread_m                7
mean_existence_rate                7
mean_edge_entropy                  7
edge_predictive_entropy            7
edge_epistemic_uncertainty         7
predicted_positive_rate            7
osm_centroid_distance_m           14
os

In [3]:
print(p3.building_id.dtype, p2.building_id.dtype)
print(p3.building_id.min(), p3.building_id.max(), p2.building_id.min(), p2.building_id.max())
print(p3.building_id.head(10).tolist(), p2.building_id.head(10).tolist())
idx=pd.read_csv(f'{P2_CACHE}/index.csv'); print(idx.columns.tolist(), idx.shape); print(idx.head())
if 'building_id' in idx: print(len(set(p3.building_id)&set(idx.building_id)), len(set(p2.building_id)&set(idx.building_id)))
m=p3.merge(p2,on='building_id'); print(len(m), (m.actual_corner_tp==m.corner_tp).mean(), (m.actual_edge_tp==m.edge_tp).mean(), (m.actual_corner_true==m.corner_true).mean())

int64 int64
31 47079 5 47058
[31, 34, 41, 56, 62, 75, 78, 83, 86, 88] [5, 6, 20, 31, 70, 75, 76, 131, 159, 192]
['building_id', 'split', 'shard', 'row', 'scale'] (31984, 5)
   building_id  split          shard  row      scale
0            3  train  shard_0000.pt    0  10.070437
1            8  train  shard_0000.pt    1   4.891159
2            9  train  shard_0000.pt    2  13.123101
3           10  train  shard_0000.pt    3  12.977716
4           14  train  shard_0000.pt    4  11.867080
3207 3235
321 0.1588785046728972 0.12149532710280374 1.0


In [4]:
s=p3.merge(idx[['building_id','split']],on='building_id',how='left')
print(s.split.value_counts(dropna=False))
print(pd.crosstab(s.coverage_class,s.split))

split
train    2580
test      321
val       306
Name: count, dtype: int64
split           test  train  val
coverage_class                  
dense            156   1326  151
sparse           165   1254  155


In [5]:
from sklearn.metrics import roc_auc_score
d=p3.merge(idx[['building_id','split']],on='building_id')
def f1(tp,p,t):
    tp,p,t=[np.asarray(x,dtype=float) for x in (tp,p,t)]
    den=p+t
    return np.where(den>0,2*tp/np.maximum(den,1),0.0)
d['corner_f1']=f1(d.actual_corner_tp,d.actual_corner_pred,d.actual_corner_true)
d['edge_f1']=f1(d.actual_edge_tp,d.actual_edge_pred,d.actual_edge_true)
d['min_f1']=d[['corner_f1','edge_f1']].min(axis=1)
d['ext_iou']=d[['osm_iou','microsoft_iou']].mean(axis=1)
print(d.groupby('split')[['corner_f1','edge_f1','min_f1']].mean())
for t in [0.5,0.6,0.7,0.8]:
    print(t, d.groupby('split')['min_f1'].apply(lambda s:(s<t).mean()).round(3).to_dict())
for sp in ['train','val','test']:
    g=d[d.split==sp]; y=(g.min_f1<0.8).astype(int)
    print(sp, round(y.mean(),3), round(roc_auc_score(y,1-g.combined_confidence),3), round(roc_auc_score(y,1-g.model_confidence),3))

       corner_f1   edge_f1    min_f1
split                               
test    0.785817  0.568032  0.564438
train   0.800126  0.577749  0.575597
val     0.799025  0.564550  0.563601
0.5 {'test': 0.408, 'train': 0.423, 'val': 0.425}
0.6 {'test': 0.573, 'train': 0.564, 'val': 0.595}
0.7 {'test': 0.695, 'train': 0.683, 'val': 0.686}
0.8 {'test': 0.782, 'train': 0.758, 'val': 0.768}
train 0.758 0.608 0.806
val 0.768 0.548 0.813
test 0.782 0.579 0.796


In [6]:
d['is_bad']=(d.min_f1<0.5).astype(int)
cols=['model_confidence','corner_confidence','edge_confidence','external_confidence','ext_iou','combined_confidence']
rows=[]
for sp,g in d.groupby('split'):
    for c in cols:
        h=g.dropna(subset=[c])
        rows.append((sp,c,len(h),round(roc_auc_score(h.is_bad,1-h[c]),3)))
print(pd.DataFrame(rows,columns=['split','score','n','auroc']).pivot(index='score',columns='split',values='auroc'))
print(d.groupby(['split','coverage_class']).is_bad.mean().round(3).unstack())
print(d.groupby('confidence_class').is_bad.mean().round(3))
print(pd.crosstab(pd.cut(d.model_confidence,[-.01,.4,.6,.7,.8,1]),d.is_bad,normalize='index').round(3))

split                 test  train    val
score                                   
combined_confidence  0.637  0.604  0.630
corner_confidence    0.775  0.743  0.703
edge_confidence      0.736  0.714  0.715
ext_iou              0.511  0.488  0.515
external_confidence  0.522  0.500  0.523
model_confidence     0.801  0.768  0.749
coverage_class  dense  sparse
split                        
test            0.365   0.448
train           0.374   0.475
val             0.404   0.445
confidence_class
high      0.103
low       0.516
medium    0.449
Name: is_bad, dtype: float64
is_bad                0      1
model_confidence              
(-0.01, 0.4]      0.000  1.000
(0.4, 0.6]        0.252  0.748
(0.6, 0.7]        0.301  0.699
(0.7, 0.8]        0.524  0.476
(0.8, 1.0]        0.849  0.151


In [7]:
d['is_bad']=(d.min_f1<0.5).astype(int)
tune=d[d.split=='train'].reset_index(drop=True)
hold=d[d.split.isin(['val','test'])].reset_index(drop=True)

def decide(df,c,r=0):
    mc=df.model_confidence.values; iou=df.ext_iou.values; av=~np.isnan(iou)
    dis=av&(iou<c['iou_lo'])
    hi=mc>=c['t_acc']; lo=mc<c['t_low']; can=r<c['max_retry']
    conds=[hi&~dis,hi&dis,lo&can,lo,dis,av]
    acts=['ACCEPT','ESCALATE','RETRY','ESCALATE','ESCALATE','SUPPLEMENT']
    why=['HIGH_CONF','EXTERNAL_DISAGREE','LOW_MODEL_CONF','RETRY_FAILED' if r>0 else 'LOW_MODEL_CONF','EXTERNAL_DISAGREE','MID_CONF_EXT_OK']
    a=np.select(conds,acts,default=c['mid_no_ext'])
    w=np.select(conds,why,default='MID_CONF_NO_EXT')
    return a,w

cfg0=dict(t_acc=0.85,t_low=0.60,iou_lo=0.70,mid_no_ext='ACCEPT_FLAG',max_retry=1)
edge=pd.DataFrame({'model_confidence':[0.85,0.8499,0.60,0.5999,0.0,0.9,0.7,0.7,0.7,0.9],
 'ext_iou':[np.nan,np.nan,np.nan,0.9,np.nan,0.3,np.nan,0.9,0.3,0.9]})
a,w=decide(edge,cfg0); edge['action']=a; edge['reason']=w
print(edge)
assert len(a)==len(edge) and set(a)<= {'ACCEPT','ACCEPT_FLAG','RETRY','SUPPLEMENT','ESCALATE'}
a2,w2=decide(edge,cfg0,r=1); print(pd.Series(a2).value_counts().to_dict())

   model_confidence  ext_iou       action             reason
0            0.8500      NaN       ACCEPT          HIGH_CONF
1            0.8499      NaN  ACCEPT_FLAG    MID_CONF_NO_EXT
2            0.6000      NaN  ACCEPT_FLAG    MID_CONF_NO_EXT
3            0.5999      0.9        RETRY     LOW_MODEL_CONF
4            0.0000      NaN        RETRY     LOW_MODEL_CONF
5            0.9000      0.3     ESCALATE  EXTERNAL_DISAGREE
6            0.7000      NaN  ACCEPT_FLAG    MID_CONF_NO_EXT
7            0.7000      0.9   SUPPLEMENT    MID_CONF_EXT_OK
8            0.7000      0.3     ESCALATE  EXTERNAL_DISAGREE
9            0.9000      0.9       ACCEPT          HIGH_CONF
{'ESCALATE': 4, 'ACCEPT_FLAG': 3, 'ACCEPT': 2, 'SUPPLEMENT': 1}


In [8]:
import itertools
P=dict(L=2.0,c_retry=0.2,c_sup=0.1,c_rev=1.0,p_fix=0.3,p_sup=0.1,budget=0.20)
def utility(df,a):
    y=df.is_bad.values
    u=np.where(np.isin(a,['ACCEPT','ACCEPT_FLAG']),-P['L']*y,0.0)
    u=np.where(a=='RETRY',-P['c_retry']-P['L']*y*(1-P['p_fix']),u)
    u=np.where(a=='SUPPLEMENT',-P['c_sup']-P['L']*y*(1-P['p_sup']),u)
    u=np.where(a=='ESCALATE',-P['c_rev'],u)
    return u.mean()
def review(a): return (a=='ESCALATE').mean()+(a=='RETRY').mean()*(1-P['p_fix'])
res=[]
for ta,tl,il,mn in itertools.product([0.75,0.8,0.85],[0.5,0.6,0.7,0.75],[0.0,0.5,0.7,0.8],['ACCEPT_FLAG','ESCALATE']):
    c=dict(t_acc=ta,t_low=tl,iou_lo=il,mid_no_ext=mn,max_retry=1)
    a,_=decide(tune,c); res.append({**c,'util':utility(tune,a),'review':review(a),**pd.Series(a).value_counts(normalize=True).to_dict()})
g=pd.DataFrame(res).fillna(0)
ok=g[(g.review<=P['budget'])&(g.ACCEPT+g.ACCEPT_FLAG<0.95)] if 'ACCEPT_FLAG' in g else g
best=ok.sort_values('util',ascending=False).head(5); print(best.round(3).T)
bc={k:best.iloc[0][k] for k in ['t_acc','t_low','iou_lo','mid_no_ext','max_retry']}
bc['max_retry']=1; json.dump({'policy':{k:(float(v) if not isinstance(v,str) else v) for k,v in bc.items()},'cost_model':P},open(f'{OUT}/policy_config.json','w'),indent=2)

                      16        17        49           48        81
t_acc               0.75      0.75       0.8          0.8      0.85
t_low                0.7       0.7       0.7          0.7       0.7
iou_lo               0.0       0.0       0.0          0.0       0.0
mid_no_ext   ACCEPT_FLAG  ESCALATE  ESCALATE  ACCEPT_FLAG  ESCALATE
max_retry              1         1         1            1         1
util              -0.789    -0.789    -0.794       -0.794    -0.802
review             0.166     0.166     0.166        0.166     0.166
ACCEPT             0.561     0.561     0.337        0.337     0.197
SUPPLEMENT         0.203     0.203     0.427        0.427     0.566
RETRY              0.236     0.236     0.236        0.236     0.236
ACCEPT_FLAG          0.0       0.0       0.0          0.0       0.0
ESCALATE             0.0       0.0       0.0          0.0     0.001


In [9]:
cfg=dict(t_acc=0.80,t_low=0.70,iou_lo=0.0,mid_no_ext='ACCEPT_FLAG',max_retry=1)
json.dump({'policy':cfg,'cost_model':P},open(f'{OUT}/policy_config.json','w'),indent=2)
a,w=decide(hold,cfg); hold['action']=a; hold['reason']=w
def firstpass(df):
    a=df.action.values; y=df.is_bad.values
    asis=np.isin(a,['ACCEPT','ACCEPT_FLAG']); flag=np.isin(a,['RETRY','ESCALATE'])
    return dict(n=len(df),accepted_as_is=asis.mean(),bad_in_accepted=y[asis].mean(),bad_caught=flag[y==1].mean(),good_flagged=flag[y==0].mean(),flagged=flag.mean())
print(pd.DataFrame({s:firstpass(g) for s,g in hold.groupby('split')}).round(3))
print(pd.crosstab(hold.action,hold.is_bad,margins=True))
for ta in [.75,.8,.85]:
    m=tune.model_confidence>=ta; print(ta,round(m.mean(),3),round(tune.is_bad[m].mean(),3))

                    test      val
n                321.000  306.000
accepted_as_is     0.333    0.350
bad_in_accepted    0.103    0.159
bad_caught         0.412    0.369
good_flagged       0.137    0.142
flagged            0.249    0.239
is_bad        0    1  All
action                   
ACCEPT      186   28  214
RETRY        51  102  153
SUPPLEMENT  129  131  260
All         366  261  627
0.75 0.561 0.245
0.8 0.337 0.155
0.85 0.197 0.128


In [10]:
import re
src=open('/kaggle/input/datasets/risha8750/phase2-core3/phase2_core.py').read()
print('\n'.join(re.findall(r'^(?:def|class) .*',src,re.M)))

def seed_everything(seed=42):
def file_key(name):
def archive_inventory(root):
def choose_archives(table):
def read_id_list(path):
def find_split_manifest(root):
def member_map(zf,suffix):
def read_xyz(zf,name):
def read_wireframe(zf,name):
def sample_points(raw,count,rng):
def normalize_sample(raw,corners,count,rng):
def build_cache(source_root,cache_root,point_count=2048,max_corners=64,max_edges=128,shard_size=256,seed=42):
class RoofDataset(Dataset):
def square_distance(src,dst):
def index_points(points,idx):
def farthest_point_sample(xyz,npoint):
def query_neighbors(radius,nsample,xyz,new_xyz):
class SetAbstraction(nn.Module):
class FeaturePropagation(nn.Module):
class CornerPointNet2(nn.Module):
def corner_targets(points,corners,counts,scales,radius_m=0.35):
def focal_binary(logits,targets,alpha=0.75,gamma=2.0):
def corner_loss(logits,offsets,pred_offsets,labels):
def predict_corners(points,logits,offsets,scale,score_thresh=0.5,radius_m=0.35,cluster_m=0.25,min_samples=2,max_corner

In [11]:
import inspect, importlib.util
spec=importlib.util.spec_from_file_location('phase2_core','/kaggle/input/datasets/risha8750/phase2-core3/phase2_core.py')
core=importlib.util.module_from_spec(spec); spec.loader.exec_module(core)
for f in [core.predict_corners,core.reconstruction_counts,core.finalize_counts,core.RoofDataset.__getitem__,core.CornerPointNet2.forward,core.EdgeGraphNet.forward]:
    print(inspect.getsource(f)); print('-----')

def predict_corners(points,logits,offsets,scale,score_thresh=0.5,radius_m=0.35,cluster_m=0.25,min_samples=2,max_corners=64):
    scores=torch.sigmoid(logits).detach().cpu().numpy()
    xyz=points[:,:3].detach().cpu().numpy()
    delta=offsets.detach().cpu().numpy()
    keep=np.flatnonzero(scores>=score_thresh)
    if len(keep)==0:
        keep=np.argsort(scores)[-4:]
    proposals=xyz[keep]+delta[keep]*(radius_m/float(scale))
    groups=DBSCAN(eps=cluster_m/float(scale),min_samples=min_samples).fit_predict(proposals)
    corners=[]
    confidence=[]
    for group in sorted(set(groups)):
        if group<0:
            continue
        mask=groups==group
        w=scores[keep][mask]
        corners.append(np.average(proposals[mask],axis=0,weights=w+1e-6))
        confidence.append(float(w.max()))
    if len(corners)<4:
        take=np.argsort(scores)[-4:]
        corners=list(xyz[take]+delta[take]*(radius_m/float(scale)))
        confidence=list(scores[take])
    order=np.argsort(confid

In [12]:
for f in [core.match_corners,core.interpolate_features]:
    print(inspect.getsource(f)); print('-----')

def match_corners(pred,truth,tolerance_m):
    if len(pred)==0 or len(truth)==0:
        return {},0,0.0
    dist=np.linalg.norm(pred[:,None]-truth[None],axis=2)
    rows,cols=linear_sum_assignment(dist)
    mapping={int(r):int(c) for r,c in zip(rows,cols) if dist[r,c]<=tolerance_m}
    total=sum(dist[r,c] for r,c in mapping.items())
    return mapping,len(mapping),float(total)

-----
def interpolate_features(points,features,corners):
    if len(corners)==0:
        return features.new_zeros((0,features.shape[-1]))
    query=torch.as_tensor(corners,dtype=points.dtype,device=points.device)[None]
    dist,idx=square_distance(query,points[None,:,:3]).topk(3,largest=False)
    weight=1/(dist+1e-8)
    weight=weight/weight.sum(-1,keepdim=True)
    return (index_points(features[None],idx)*weight.unsqueeze(-1)).sum(2).squeeze(0)

-----


In [13]:
import glob, json, re
hits=[p for p in glob.glob('/kaggle/input/**/*',recursive=True) if p.endswith(('.ipynb','.py')) and 'phase2_core' not in p]
print(hits)
def cells(p):
    if p.endswith('.ipynb'):
        return [''.join(c['source']) for c in json.load(open(p))['cells'] if c['cell_type']=='code']
    return [open(p).read()]
for p in hits:
    for i,s in enumerate(cells(p)):
        if 'predict_corners(' in s and 'reconstruction_counts' in s: print('EVAL',p,i); print(s[:6000]); print('=====')
        if re.search(r'dropout|\.train\(\)',s) and 'confidence' in s: print('MC',p,i); print(s[:6000]); print('=====')

['/kaggle/input/notebooks/haruistaken/phase3-alt/__notebook__.ipynb', '/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/__notebook__.ipynb']
EVAL /kaggle/input/notebooks/haruistaken/phase3-alt/__notebook__.ipynb 2
def square_distance(src, dst):
    return torch.sum(src ** 2, -1, keepdim=True) + torch.sum(dst ** 2, -1).unsqueeze(1) - 2 * torch.matmul(src, dst.transpose(1, 2))


def index_points(points, idx):
    batch = torch.arange(points.shape[0], device=points.device).view(-1, *([1] * (idx.ndim - 1))).expand_as(idx)
    return points[batch, idx]


def farthest_point_sample(xyz, npoint):
    b, n, _ = xyz.shape
    centroids = torch.zeros(b, npoint, dtype=torch.long, device=xyz.device)
    distance = torch.full((b, n), 1e10, device=xyz.device)
    farthest = torch.randint(0, n, (b,), device=xyz.device)
    batch = torch.arange(b, device=xyz.device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch, farthest].view(b, 1, 3)
       

In [14]:
import glob, hashlib
idx3=glob.glob('/kaggle/input/**/phase2_cache/index.csv',recursive=True); print(idx3)
ck=glob.glob('/kaggle/input/**/*_best.pt',recursive=True)
md5=lambda p: hashlib.md5(open(p,'rb').read()).hexdigest()
for p in ck: print(p, md5(p))
for p in idx3:
    i=pd.read_csv(p); print(p, i.split.value_counts().to_dict(), set(p3.building_id)<=set(i.building_id[i.split=='test']))

['/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache/index.csv']
/kaggle/input/datasets/risha8750/phase2-latest-outputs/phase2_outputs/corner/corner_best.pt c83e8c7e6b549a6484d906ef1438dfa2
/kaggle/input/datasets/risha8750/phase2-latest-outputs/phase2_outputs/edge/edge_best.pt 87efc29cc895536659cd54c685625838
/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache/index.csv {'train': 25579, 'test': 3235, 'val': 3170} False


In [15]:
from sklearn.model_selection import train_test_split
d['is_bad']=(d.min_f1<0.5).astype(int)
tune,hold=train_test_split(d,test_size=0.5,stratify=d.is_bad.astype(str)+d.coverage_class,random_state=42)
tune,hold=tune.reset_index(drop=True),hold.reset_index(drop=True)
print(len(tune),len(hold),set(tune.building_id).isdisjoint(hold.building_id))
print(pd.crosstab(tune.coverage_class,tune.is_bad,normalize='all').round(3))
print(pd.crosstab(hold.coverage_class,hold.is_bad,normalize='all').round(3))

res=[]
for ta,tl,il,mn in itertools.product([0.75,0.8,0.85],[0.5,0.6,0.7,0.75],[0.0,0.5,0.7,0.8],['ACCEPT_FLAG','ESCALATE']):
    c=dict(t_acc=ta,t_low=tl,iou_lo=il,mid_no_ext=mn,max_retry=1)
    a,_=decide(tune,c)
    res.append({**c,'util':utility(tune,a),'review':review(a),**pd.Series(a).value_counts(normalize=True).to_dict()})
g=pd.DataFrame(res).fillna(0)
print(g[g.review<=P['budget']].sort_values('util',ascending=False).head(5).round(3).T)

a,w=decide(hold,cfg); hold['action']=a; hold['reason']=w
print(pd.Series(firstpass(hold)).round(3))
print(pd.crosstab(hold.action,hold.is_bad,margins=True))
rng=np.random.default_rng(42); asis=np.isin(a,['ACCEPT','ACCEPT_FLAG']); y=hold.is_bad.values[asis]
bs=[y[rng.integers(0,len(y),len(y))].mean() for _ in range(1000)]
print(np.percentile(bs,[2.5,97.5]).round(3))

1603 1604 True
is_bad              0      1
coverage_class              
dense           0.318  0.192
sparse          0.260  0.231
is_bad              0      1
coverage_class              
dense           0.318  0.191
sparse          0.261  0.230
                      16        17        49           48        81
t_acc               0.75      0.75       0.8          0.8      0.85
t_low                0.7       0.7       0.7          0.7       0.7
iou_lo               0.0       0.0       0.0          0.0       0.0
mid_no_ext   ACCEPT_FLAG  ESCALATE  ESCALATE  ACCEPT_FLAG  ESCALATE
max_retry              1         1         1            1         1
util              -0.789    -0.789    -0.795       -0.795    -0.802
review             0.164     0.164     0.164        0.164     0.165
ACCEPT             0.552     0.552     0.336        0.336     0.198
SUPPLEMENT         0.214     0.214      0.43         0.43     0.567
RETRY              0.234     0.234     0.234        0.234     0.234
ACCEP

In [16]:
import os, json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
P2_CACHE='/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache'
P3_CSV='/kaggle/input/datasets/risha8750/phase3-results-csv/phase3_results.csv'
OUT='/kaggle/working/phase4_decision'
for s in ['','plots','retry_outputs','meshes_v2']: os.makedirs(f'{OUT}/{s}',exist_ok=True)

p3=pd.read_csv(P3_CSV); idx=pd.read_csv(f'{P2_CACHE}/index.csv')
d=p3.merge(idx[['building_id','split']],on='building_id')
def f1(tp,p,t):
    tp,p,t=[np.asarray(x,dtype=float) for x in (tp,p,t)]
    return np.where(p+t>0,2*tp/np.maximum(p+t,1),0.0)
d['corner_f1']=f1(d.actual_corner_tp,d.actual_corner_pred,d.actual_corner_true)
d['edge_f1']=f1(d.actual_edge_tp,d.actual_edge_pred,d.actual_edge_true)
d['min_f1']=d[['corner_f1','edge_f1']].min(axis=1)
d['ext_iou']=d[['osm_iou','microsoft_iou']].mean(axis=1)
d['is_bad']=(d.min_f1<0.5).astype(int)

def decide(df,c,r=0):
    mc=df.model_confidence.values; iou=df.ext_iou.values; av=~np.isnan(iou)
    dis=av&(iou<c['iou_lo'])
    hi=mc>=c['t_acc']; lo=mc<c['t_low']; can=r<c['max_retry']
    conds=[hi&~dis,hi&dis,lo&can,lo,dis,av]
    acts=['ACCEPT','ESCALATE','RETRY','ESCALATE','ESCALATE','SUPPLEMENT']
    why=['HIGH_CONF','EXTERNAL_DISAGREE','LOW_MODEL_CONF','RETRY_FAILED' if r>0 else 'LOW_MODEL_CONF','EXTERNAL_DISAGREE','MID_CONF_EXT_OK']
    return np.select(conds,acts,default=c['mid_no_ext']),np.select(conds,why,default='MID_CONF_NO_EXT')

cfg=dict(t_acc=0.80,t_low=0.70,iou_lo=0.0,mid_no_ext='ACCEPT_FLAG',max_retry=1)
tune,hold=train_test_split(d,test_size=0.5,stratify=d.is_bad.astype(str)+d.coverage_class,random_state=42)
tune,hold=tune.reset_index(drop=True),hold.reset_index(drop=True)
tune['half']='tune'; hold['half']='hold'
for g in (tune,hold):
    a,w=decide(g,cfg); g['action']=a; g['reason']=w
fp=pd.concat([tune,hold])[['building_id','half','coverage_class','model_confidence','combined_confidence','ext_iou','corner_f1','edge_f1','min_f1','is_bad','action','reason']]
fp.to_csv(f'{OUT}/phase4_first_pass.csv',index=False)
json.dump({'policy':cfg},open(f'{OUT}/policy_config.json','w'),indent=2)
print(fp.groupby(['half','action']).size().unstack())

action  ACCEPT  RETRY  SUPPLEMENT
half                             
hold       545    388         671
tune       538    375         690


In [17]:
for n in ['p3','idx','FIRST_PASS','index_by_id','load_item','score_building','BASE','corner_model','edge_model']:
    print(n, n in globals())

p3 True
idx True
FIRST_PASS False
index_by_id False
load_item False
score_building False
BASE False
corner_model False
edge_model False


In [18]:
import os, sys, time, hashlib, importlib.util
import numpy as np, pandas as pd, torch
from sklearn.cluster import DBSCAN

P2_CACHE='/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache'
P3_CSV='/kaggle/input/datasets/risha8750/phase3-results-csv/phase3_results.csv'
CORE='/kaggle/input/datasets/risha8750/phase2-core3/phase2_core.py'
P2_OUT='/kaggle/input/datasets/risha8750/phase2-latest-outputs/phase2_outputs'
OUT='/kaggle/working/phase4_decision'
os.makedirs(f'{OUT}/retry_outputs',exist_ok=True)

p3=pd.read_csv(P3_CSV); idx=pd.read_csv(f'{P2_CACHE}/index.csv'); index_by_id=idx.set_index('building_id')

spec=importlib.util.spec_from_file_location('phase2_core',CORE)
core=importlib.util.module_from_spec(spec); sys.modules['phase2_core']=core; spec.loader.exec_module(core)
core.seed_everything(42)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device',DEVICE)

def load_weights(path,model):
    state=torch.load(path,map_location=DEVICE,weights_only=False)
    raw=state['model'] if isinstance(state,dict) and 'model' in state else state
    r=model.load_state_dict(raw,strict=False)
    print(path.split('/')[-1],'missing',r.missing_keys,'unexpected',r.unexpected_keys)

corner_model=core.CornerPointNet2().to(DEVICE); load_weights(f'{P2_OUT}/corner/corner_best.pt',corner_model)
edge_model=core.EdgeGraphNet().to(DEVICE); load_weights(f'{P2_OUT}/edge/edge_best.pt',edge_model)
corner_model.eval(); edge_model.eval()

_sh={'name':None,'data':None}
def load_item(bid):
    r=index_by_id.loc[int(bid)]
    if _sh['name']!=r.shard:
        _sh['data']=torch.load(f'{P2_CACHE}/{r.shard}',map_location='cpu',weights_only=False); _sh['name']=r.shard
    return _sh['data'][int(r.row)]

def denorm(p,center,scale): return p*float(scale)+np.asarray(center,dtype=np.float64)

def predict_corners_p3(points,logits,offsets,scale,score_thresh,cluster_m,min_samples,offset_norm_m=0.35,max_corners=64):
    scores=torch.sigmoid(logits).detach().cpu().numpy()
    xyz=points[:,:3].detach().cpu().numpy()
    delta=offsets.detach().cpu().numpy()
    empty=(np.zeros((0,3),dtype=np.float32),np.zeros((0,),dtype=np.float32))
    keep=np.flatnonzero(scores>=score_thresh)
    if len(keep)==0: return empty
    prop=xyz[keep]+delta[keep]*(offset_norm_m/float(scale))
    groups=DBSCAN(eps=cluster_m/float(scale),min_samples=min_samples).fit_predict(prop)
    cs,cf=[],[]
    for g in sorted(set(groups)):
        if g<0: continue
        m=groups==g; w=scores[keep][m]
        cs.append(np.average(prop[m],axis=0,weights=w+1e-6)); cf.append(float(w.max()))
    if not cs: return empty
    o=np.argsort(cf)[::-1][:max_corners]
    return np.asarray(cs,dtype=np.float32)[o],np.asarray(cf,dtype=np.float32)[o]

def perturb(points,jit_norm,drop,rng):
    points=points.clone(); n=points.shape[0]; nd=int(drop*n)
    if nd>0:
        keep=rng.choice(n,n-nd,replace=False); refill=rng.choice(keep,nd,replace=True)
        points=points[torch.as_tensor(np.concatenate([keep,refill]),dtype=torch.long)]
    if jit_norm>0: points[:,:3]+=torch.randn_like(points[:,:3])*jit_norm
    return points

def multipass(item,prm,rng):
    scale=float(item['scale']); center=item['center'].numpy(); jn=prm['jitter_std_m']/scale
    base=item['points'].float(); out=[]
    with torch.no_grad():
        for p in range(prm['n_passes']):
            pts=base if p==0 else perturb(base,jn,prm['drop_fraction'],rng)
            pts=pts.unsqueeze(0).to(DEVICE)
            lg,of,sh=corner_model(pts)
            pn,ps=predict_corners_p3(pts[0],lg[0],of[0],scale,prm['score_thresh'],prm['cluster_m'],prm['min_samples'])
            out.append(dict(pred_norm=pn,pred_scores=ps,pred_m=denorm(pn,center,scale),shared=sh[0].cpu() if p==0 else None))
    return out

def corner_unc(passes,tol):
    ref=passes[0]['pred_m']; n=len(ref)
    if n==0: return dict(corner_confidence=0.0)
    samples=[[] for _ in range(n)]; ex=np.zeros(n)
    for p in passes:
        mp,_,_=core.match_corners(ref,p['pred_m'],tol)
        for r,c in mp.items():
            samples[r].append(p['pred_m'][c]); ex[r]+=1
    stds=[float(np.std(np.array(s),axis=0).mean()) for s in samples if len(s)>=2]
    ms=float(np.mean(stds)) if stds else 0.0
    return dict(corner_confidence=float(np.exp(-ms/0.5)*(ex/len(passes)).mean()))

def ent(p):
    p=np.clip(p,1e-6,1-1e-6); return -(p*np.log2(p)+(1-p)*np.log2(1-p))

def edge_unc(ref,points,n_passes):
    pn,ps,sh=ref['pred_norm'],ref['pred_scores'],ref['shared']
    if len(pn)<2: return dict(edge_confidence=0.0),np.zeros((0,2),dtype=int)
    cxyz=torch.as_tensor(pn,dtype=torch.float32,device=DEVICE); csc=torch.as_tensor(ps,dtype=torch.float32,device=DEVICE)
    feats=core.interpolate_features(points.float().to(DEVICE),sh.to(DEVICE),pn)
    edge_model.train(); pr=[]
    with torch.no_grad():
        for _ in range(n_passes):
            pairs,lg=edge_model(cxyz,csc,feats); pr.append(torch.sigmoid(lg).cpu().numpy())
    edge_model.eval()
    pr=np.stack(pr); mp=pr.mean(0)
    pe=ent(mp); ee=ent(pr).mean(0); mi=np.clip(pe-ee,0,None)
    pos=mp>0.5
    ec=float(np.clip(0.7*(1-float(pe.mean()))+0.3*(1-float(mi.mean())),0,1))
    return dict(edge_confidence=ec),pairs.cpu().numpy()[pos]

BASE=dict(score_thresh=0.5,min_samples=1,cluster_m=0.25,n_passes=8,jitter_std_m=0.05,drop_fraction=0.08,tol_m=1.0,seed_tag='')

def score_building(item,prm):
    bid=int(item['building_id'])
    sd=int.from_bytes(hashlib.sha256((str(bid)+prm['seed_tag']).encode()).digest()[:4],'little')
    rng=np.random.default_rng(sd); torch.manual_seed(sd)
    passes=multipass(item,prm,rng)
    cu=corner_unc(passes,prm['tol_m'])
    eu,edges=edge_unc(passes[0],item['points'],prm['n_passes'])
    scale=float(item['scale']); center=item['center'].numpy()
    n=int(item['corner_count']); ne=int(item['edge_count'])
    truth=denorm(item['corners'][:n].numpy(),center,scale)
    c=core.reconstruction_counts(passes[0]['pred_m'],edges,truth,item['edges'][:ne].numpy(),tolerance_m=prm['tol_m'])
    cf=2*c['corner_tp']/max(c['corner_pred']+c['corner_true'],1)
    ef=2*c['edge_tp']/max(c['edge_pred']+c['edge_true'],1)
    return dict(building_id=bid,corner_confidence=cu['corner_confidence'],edge_confidence=eu['edge_confidence'],
        model_confidence=0.5*cu['corner_confidence']+0.5*eu['edge_confidence'],n_corners=len(passes[0]['pred_m']),
        corner_f1=cf,edge_f1=ef,min_f1=min(cf,ef))

sp=f'{OUT}/phase4_scores.csv'
all_ids=sorted(p3.building_id.astype(int).tolist(),key=lambda b:(index_by_id.loc[b].shard,index_by_id.loc[b].row))
rows=pd.read_csv(sp).to_dict('records') if os.path.exists(sp) else []
seen={int(r['building_id']) for r in rows}
todo=[b for b in all_ids if b not in seen]
print('already scored',len(seen),'| to do',len(todo))
t0=time.time()
for i,b in enumerate(todo):
    rows.append(score_building(load_item(b),BASE))
    if (i+1)%100==0:
        pd.DataFrame(rows).to_csv(sp,index=False)
        print(len(rows),'of',len(all_ids),round((time.time()-t0)/60,1),'min')
pd.DataFrame(rows).to_csv(sp,index=False); print('done',len(rows))

device cuda
corner_best.pt missing [] unexpected []
edge_best.pt missing [] unexpected []
already scored 0 | to do 3207
100 of 3207 1.3 min
200 of 3207 2.6 min
300 of 3207 3.9 min
400 of 3207 5.1 min
500 of 3207 6.4 min
600 of 3207 7.7 min
700 of 3207 9.0 min
800 of 3207 10.2 min
900 of 3207 11.5 min
1000 of 3207 12.8 min
1100 of 3207 14.1 min
1200 of 3207 15.3 min
1300 of 3207 16.6 min
1400 of 3207 17.9 min
1500 of 3207 19.2 min
1600 of 3207 20.4 min
1700 of 3207 21.7 min
1800 of 3207 23.0 min
1900 of 3207 24.3 min
2000 of 3207 25.5 min
2100 of 3207 26.8 min
2200 of 3207 28.1 min
2300 of 3207 29.3 min
2400 of 3207 30.6 min
2500 of 3207 31.9 min
2600 of 3207 33.2 min
2700 of 3207 34.5 min
2800 of 3207 35.7 min
2900 of 3207 37.0 min
3000 of 3207 38.3 min
3100 of 3207 39.6 min
3200 of 3207 40.8 min
done 3207


In [19]:
m=pd.DataFrame(rows).merge(p3[['building_id','model_confidence','corner_confidence','edge_confidence','num_predicted_corners','actual_corner_true']],on='building_id',suffixes=('_new','_csv'))
m['f1_csv']=fp.set_index('building_id').loc[m.building_id,'min_f1'].values
print(m[['model_confidence_new','model_confidence_csv','corner_confidence_new','corner_confidence_csv','edge_confidence_new','edge_confidence_csv','n_corners','num_predicted_corners','min_f1','f1_csv']].agg(['mean','std']).round(3).T)
print('same predicted corner count:',(m.n_corners==m.num_predicted_corners).mean().round(3))
tc=np.array([int(load_item(b)['corner_count']) for b in m.building_id])
print('same true corner count:',(tc==m.actual_corner_true.values).mean().round(3))
print(m[['building_id','n_corners','num_predicted_corners','min_f1','f1_csv']].head(8))

                         mean     std
model_confidence_new    0.677   0.096
model_confidence_csv    0.764   0.099
corner_confidence_new   0.750   0.114
corner_confidence_csv   0.718   0.134
edge_confidence_new     0.605   0.108
edge_confidence_csv     0.809   0.094
n_corners              11.198   6.936
num_predicted_corners  15.087  12.034
min_f1                  0.433   0.253
f1_csv                  0.573   0.265
same predicted corner count: 0.136
same true corner count: 1.0
   building_id  n_corners  num_predicted_corners    min_f1    f1_csv
0           34          4                      4  1.000000  1.000000
1           41          3                      6  0.666667  0.400000
2           56          7                      4  0.260870  0.142857
3           62          9                      5  0.160000  0.266667
4           78          4                      6  0.600000  1.000000
5           83          9                     10  0.782609  0.916667
6           86         14           

In [20]:
for n in ['corner/corner_best.pt','corner/corner_last.pt','edge/edge_best.pt','edge/edge_last.pt']:
    s=torch.load(f'{P2_OUT}/{n}',map_location='cpu',weights_only=False)
    h=s.get('history',[]) if isinstance(s,dict) else []
    print(n,'epoch',s.get('epoch') if isinstance(s,dict) else None,'| history',len(h),'|',h[-1] if h else None)

csvn=m.set_index('building_id').num_predicted_corners
tst=m.building_id.tolist()[:20]
for name in ['corner_best.pt','corner_last.pt']:
    s=torch.load(f'{P2_OUT}/corner/{name}',map_location=DEVICE,weights_only=False)
    corner_model.load_state_dict(s['model'] if 'model' in s else s); corner_model.eval()
    got=[len(multipass(load_item(b),{**BASE,'n_passes':1},np.random.default_rng(0))[0]['pred_m']) for b in tst]
    print(name,'mean corners',round(np.mean(got),2),'| csv mean',round(csvn.loc[tst].mean(),2),'| corr',round(np.corrcoef(got,csvn.loc[tst])[0,1],3))
s=torch.load(f'{P2_OUT}/corner/corner_best.pt',map_location=DEVICE,weights_only=False)
_=corner_model.eval()

corner/corner_best.pt epoch 19 | history 20 | {'epoch': 19, 'train_loss': np.float64(0.055312054057863574), 'train_cls': np.float64(0.007033376351364591), 'train_offset': np.float64(0.04827867775942248), 'val_loss': np.float64(0.05421943516596823), 'val_cls': np.float64(0.006845041225908165), 'val_offset': np.float64(0.04737439395665934), 'lr': 5.4999999999999995e-05}
corner/corner_last.pt epoch 19 | history 20 | {'epoch': 19, 'train_loss': np.float64(0.055312054057863574), 'train_cls': np.float64(0.007033376351364591), 'train_offset': np.float64(0.04827867775942248), 'val_loss': np.float64(0.05421943516596823), 'val_cls': np.float64(0.006845041225908165), 'val_offset': np.float64(0.04737439395665934), 'lr': 5.4999999999999995e-05}
edge/edge_best.pt epoch 7 | history 8 | {'epoch': 7, 'train_loss': 0.0167485819844415, 'val_loss': 0.016266808015138662, 'lr': 8.511087728614862e-05}
edge/edge_last.pt epoch 19 | history 20 | {'epoch': 19, 'train_loss': 4.526853848987255, 'val_loss': 4.56301

In [21]:
import glob, hashlib
tst=fp.building_id.astype(int).sample(20,random_state=1).tolist()
csvn=p3.set_index('building_id').num_predicted_corners
csv_ec=p3.set_index('building_id').edge_confidence
def ncorners(model,b):
    it=load_item(b); pts=it['points'].float().unsqueeze(0).to(DEVICE)
    with torch.no_grad(): lg,of,_=model(pts)
    pn,_=predict_corners_p3(pts[0],lg[0],of[0],float(it['scale']),0.7,0.25,1)
    return len(pn)
rows=[]
for p in sorted(glob.glob('/kaggle/input/**/*.pt',recursive=True)):
    if 'shard_' in p: continue
    try: s=torch.load(p,map_location='cpu',weights_only=False)
    except Exception as e: rows.append(dict(path=p[-70:],note=str(e)[:30])); continue
    st=s['model'] if isinstance(s,dict) and 'model' in s else s
    keys=list(st.keys()) if isinstance(st,dict) else []
    kind='corner' if any('corner_head' in k for k in keys) else 'edge' if any('edge_head' in k for k in keys) else '?'
    r=dict(path=p[-70:],kind=kind,epoch=s.get('epoch') if isinstance(s,dict) else None,md5=hashlib.md5(open(p,'rb').read()).hexdigest()[:8])
    if kind=='corner':
        m2=core.CornerPointNet2().to(DEVICE); m2.load_state_dict(st,strict=False); m2.eval()
        r['mean_corners']=round(float(np.mean([ncorners(m2,b) for b in tst])),2)
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
print('csv mean corners on these 20:',round(float(csvn.loc[tst].mean()),2))

for name in ['edge_best.pt','edge_last.pt']:
    s=torch.load(f'{P2_OUT}/edge/{name}',map_location=DEVICE,weights_only=False)
    edge_model.load_state_dict(s['model'] if 'model' in s else s); edge_model.eval()
    out=[score_building(load_item(b),BASE) for b in tst]
    print(name,'mean edge_conf',round(np.mean([o['edge_confidence'] for o in out]),3),'| csv',round(float(csv_ec.loc[tst].mean()),3),'| mean min_f1',round(np.mean([o['min_f1'] for o in out]),3))
s=torch.load(f'{P2_OUT}/edge/edge_best.pt',map_location=DEVICE,weights_only=False)
edge_model.load_state_dict(s['model'] if 'model' in s else s); _=edge_model.eval()

                                                                  path   kind  epoch      md5  mean_corners
s/risha8750/phase2-latest-outputs/phase2_outputs/corner/corner_best.pt corner     19 c83e8c7e          1.75
s/risha8750/phase2-latest-outputs/phase2_outputs/corner/corner_last.pt corner     19 5957db87          1.55
asets/risha8750/phase2-latest-outputs/phase2_outputs/edge/edge_best.pt   edge      7 87efc29c           NaN
asets/risha8750/phase2-latest-outputs/phase2_outputs/edge/edge_last.pt   edge     19 41cc475a           NaN
csv mean corners on these 20: 16.05
edge_best.pt mean edge_conf 0.579 | csv 0.822 | mean min_f1 0.397
edge_last.pt mean edge_conf 0.516 | csv 0.822 | mean min_f1 0.16


In [22]:
print(pd.read_csv(f'{P2_OUT}/test_metrics.csv').round(3))
ids=fp.sample(40,random_state=1).building_id.astype(int).tolist()
res=[]
for th in [0.05,0.1,0.2,0.3,0.5,0.7]:
    nc,f=[],[]
    for b in ids:
        it=load_item(b); sc=float(it['scale']); ce=it['center'].numpy()
        pts=it['points'].float().unsqueeze(0).to(DEVICE)
        with torch.no_grad(): lg,of,_=corner_model(pts)
        pn,_=predict_corners_p3(pts[0],lg[0],of[0],sc,th,0.25,1)
        pm=denorm(pn,ce,sc); n=int(it['corner_count']); tr=denorm(it['corners'][:n].numpy(),ce,sc)
        _,tp,_=core.match_corners(pm,tr,1.0)
        nc.append(len(pm)); f.append(2*tp/max(len(pm)+n,1))
    res.append((th,np.mean(nc),np.mean(f)))
print(pd.DataFrame(res,columns=['thresh','mean_pred_corners','corner_f1_1m']).round(3))
print('true mean corners',np.mean([int(load_item(b)['corner_count']) for b in ids]),'| CSV corner_f1 mean',round(fp.set_index('building_id').loc[ids,'corner_f1'].mean(),3))

   tolerance_m    ACO     CP     CR   C_F1     EP     ER   E_F1
0         0.10  0.070  0.146  0.056  0.081  0.012  0.006  0.008
1         0.25  0.135  0.461  0.178  0.257  0.088  0.048  0.062
2         0.50  0.191  0.629  0.242  0.350  0.147  0.080  0.104
   thresh  mean_pred_corners  corner_f1_1m
0    0.05             63.300         0.313
1    0.10             62.925         0.315
2    0.20             50.725         0.378
3    0.30             39.500         0.505
4    0.50             11.725         0.692
5    0.70              1.825         0.283
true mean corners 15.125 | CSV corner_f1 mean 0.799


In [23]:
from sklearn.metrics import roc_auc_score
tune_ids=fp[fp.half=='tune'].building_id.astype(int).sample(50,random_state=7).tolist()
tune_ids=sorted(tune_ids,key=lambda b:(index_by_id.loc[b].shard,index_by_id.loc[b].row))
out=[]
for en in ['edge_best.pt','edge_last.pt']:
    load_weights(f'{P2_OUT}/edge/{en}',edge_model); edge_model.eval()
    for th in [0.4,0.5,0.6]:
        prm={**BASE,'score_thresh':th,'n_passes':4,'seed_tag':'sweep'}
        r=pd.DataFrame([score_building(load_item(b),prm) for b in tune_ids])
        y=(r.min_f1<0.5).astype(int)
        au=roc_auc_score(y,1-r.model_confidence) if 0<y.sum()<len(y) else np.nan
        out.append(dict(edge=en,thresh=th,corners=r.n_corners.mean(),corner_f1=r.corner_f1.mean(),edge_f1=r.edge_f1.mean(),min_f1=r.min_f1.mean(),bad_rate=y.mean(),conf_auroc=au))
print(pd.DataFrame(out).round(3))
load_weights(f'{P2_OUT}/edge/edge_best.pt',edge_model); _=edge_model.eval()

edge_best.pt missing [] unexpected []
edge_last.pt missing [] unexpected []
           edge  thresh  corners  corner_f1  edge_f1  min_f1  bad_rate  \
0  edge_best.pt     0.4    30.26      0.602    0.253   0.253      0.86   
1  edge_best.pt     0.5    13.66      0.669    0.352   0.352      0.78   
2  edge_best.pt     0.6     5.60      0.505    0.241   0.241      0.90   
3  edge_last.pt     0.4    30.26      0.602    0.060   0.060      1.00   
4  edge_last.pt     0.5    13.66      0.669    0.095   0.095      0.98   
5  edge_last.pt     0.6     5.60      0.505    0.102   0.102      1.00   

   conf_auroc  
0       0.738  
1       0.734  
2       0.578  
3         NaN  
4       0.367  
5         NaN  
edge_best.pt missing [] unexpected []


In [24]:
BASE['score_thresh']=0.5
load_weights(f'{P2_OUT}/edge/edge_best.pt',edge_model); _=edge_model.eval()
sp=f'{OUT}/phase4_scores.csv'
all_ids=sorted(p3.building_id.astype(int).tolist(),key=lambda b:(index_by_id.loc[b].shard,index_by_id.loc[b].row))
rows=pd.read_csv(sp).to_dict('records') if os.path.exists(sp) else []
seen={int(r['building_id']) for r in rows}
todo=[b for b in all_ids if b not in seen]
t0=time.time()
for i,b in enumerate(todo):
    rows.append(score_building(load_item(b),BASE))
    if (i+1)%100==0:
        pd.DataFrame(rows).to_csv(sp,index=False)
        print(len(rows),'of',len(all_ids),round((time.time()-t0)/60,1),'min')
pd.DataFrame(rows).to_csv(sp,index=False); print('done',len(rows))

edge_best.pt missing [] unexpected []
done 3207


In [25]:
RETRY_CFGS={'A':dict(score_thresh=0.6,seed_tag='r1'),
            'B':dict(score_thresh=0.5,seed_tag='r1'),
            'C':dict(score_thresh=0.5,cluster_m=0.35,n_passes=16,seed_tag='r1')}
pool=fp[(fp.half=='tune')&(fp.action=='RETRY')].building_id.astype(int)
ids=pool.sample(min(150,len(pool)),random_state=42).tolist()
ids=sorted(ids,key=lambda b:(index_by_id.loc[b].shard,index_by_id.loc[b].row))
sel=[]
for b in ids:
    it=load_item(b); o=score_building(it,BASE)
    for k,cfg_r in RETRY_CFGS.items():
        n=score_building(it,{**BASE,**cfg_r})
        sel.append(dict(building_id=b,cfg=k,orig_mc=o['model_confidence'],new_mc=n['model_confidence'],orig_f1=o['min_f1'],new_f1=n['min_f1']))
sel=pd.DataFrame(sel); sel.to_csv(f'{OUT}/retry_outputs/retry_config_selection.csv',index=False)
def summ(d_):
    dF=d_.new_f1-d_.orig_f1; kept=np.where(d_.new_mc>d_.orig_mc,d_.new_f1,d_.orig_f1)
    return pd.Series(dict(n=len(d_),improved=(dF>0.02).mean(),same=(dF.abs()<=0.02).mean(),worse=(dF<-0.02).mean(),
        mean_dF1=dF.mean(),mean_dMC=(d_.new_mc-d_.orig_mc).mean(),share_kept=(d_.new_mc>d_.orig_mc).mean(),kept_gain=(kept-d_.orig_f1).mean()))
tab=sel.groupby('cfg').apply(summ)
print(tab.round(3))
BEST=tab.kept_gain.idxmax(); print('BEST',BEST)

         n  improved   same  worse  mean_dF1  mean_dMC  share_kept  kept_gain
cfg                                                                          
A    150.0     0.173  0.140  0.687    -0.127    -0.008       0.600     -0.060
B    150.0     0.347  0.287  0.367    -0.016    -0.006       0.427     -0.005
C    150.0     0.453  0.200  0.347     0.023    -0.008       0.467      0.027
BEST C


/tmp/ipykernel_49/2817257440.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tab=sel.groupby('cfg').apply(summ)


In [26]:
ids_all=fp[fp.action=='RETRY'].building_id.astype(int).tolist()
ids_all=sorted(ids_all,key=lambda b:(index_by_id.loc[b].shard,index_by_id.loc[b].row))
rp=f'{OUT}/retry_outputs/retry_results.csv'
done=pd.read_csv(rp) if os.path.exists(rp) else pd.DataFrame()
seen=set(done.building_id) if len(done) else set()
rows=done.to_dict('records') if len(done) else []
half_of=fp.set_index('building_id').half.to_dict()
for i,b in enumerate([x for x in ids_all if x not in seen]):
    it=load_item(b); o=score_building(it,BASE); n=score_building(it,{**BASE,**RETRY_CFGS[BEST]})
    keep_new=n['model_confidence']>o['model_confidence']
    rows.append(dict(building_id=b,half=half_of[b],cfg=BEST,orig_mc=o['model_confidence'],new_mc=n['model_confidence'],
        orig_f1=o['min_f1'],new_f1=n['min_f1'],orig_corners=o['n_corners'],new_corners=n['n_corners'],
        kept='retry' if keep_new else 'original',final_mc=max(o['model_confidence'],n['model_confidence']),
        final_f1=n['min_f1'] if keep_new else o['min_f1']))
    if (i+1)%50==0: pd.DataFrame(rows).to_csv(rp,index=False); print(len(rows),'of',len(ids_all))
res_r=pd.DataFrame(rows); res_r.to_csv(rp,index=False)
res_r['dF1']=res_r.new_f1-res_r.orig_f1
print(res_r.groupby('half').apply(lambda x:pd.Series(dict(n=len(x),improved=(x.dF1>0.02).mean(),same=(x.dF1.abs()<=0.02).mean(),worse=(x.dF1<-0.02).mean(),kept_gain=(x.final_f1-x.orig_f1).mean()))).round(3))

50 of 763
100 of 763
150 of 763
200 of 763
250 of 763
300 of 763
350 of 763
400 of 763
450 of 763
500 of 763
550 of 763
600 of 763
650 of 763
700 of 763
750 of 763
          n  improved   same  worse  kept_gain
half                                          
hold  388.0     0.528  0.235  0.237      0.039
tune  375.0     0.496  0.219  0.285      0.040


/tmp/ipykernel_49/3360343446.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(res_r.groupby('half').apply(lambda x:pd.Series(dict(n=len(x),improved=(x.dF1>0.02).mean(),same=(x.dF1.abs()<=0.02).mean(),worse=(x.dF1<-0.02).mean(),kept_gain=(x.final_f1-x.orig_f1).mean()))).round(3))


In [27]:
import glob
for f in glob.glob(f'{OUT}/retry_outputs/*.csv'): os.rename(f,f+'.old_phase3_basis')
print(len(pd.read_csv(f'{OUT}/phase4_scores.csv')))

3207


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
s=pd.read_csv(f'{OUT}/phase4_scores.csv')
d=s.merge(p3[['building_id','coverage_class']],on='building_id')
print('rows',len(d),'unique',d.building_id.is_unique,'nans',int(d.isna().sum().sum()))
print(d[['n_corners','corner_f1','edge_f1','min_f1','model_confidence']].describe().round(3))
thr=float(d.min_f1.quantile(0.40)); d['is_bad']=(d.min_f1<=thr).astype(int)
print('label threshold',round(thr,3),'bad rate',round(d.is_bad.mean(),3))
for c in ['model_confidence','corner_confidence','edge_confidence']:
    print(c,'AUROC',round(roc_auc_score(d.is_bad,1-d[c]),3))
tune,hold=train_test_split(d,test_size=0.5,stratify=d.is_bad.astype(str)+d.coverage_class,random_state=42)
d['half']=np.where(d.building_id.isin(tune.building_id),'tune','hold')
print(d.groupby('half').size().to_dict(),set(tune.building_id).isdisjoint(hold.building_id))
print(d.groupby(['half','coverage_class']).is_bad.mean().round(3).unstack())
print(pd.crosstab(pd.qcut(d.model_confidence,10,duplicates='drop'),d.is_bad,normalize='index').round(3))
d.to_csv(f'{OUT}/phase4_labeled.csv',index=False)

rows 3207 unique True nans 0
       n_corners  corner_f1   edge_f1    min_f1  model_confidence
count   3207.000   3207.000  3207.000  3207.000          3207.000
mean      11.198      0.708     0.433     0.433             0.677
std        6.936      0.176     0.253     0.253             0.096
min        0.000      0.000     0.000     0.000             0.000
25%        6.000      0.600     0.250     0.250             0.621
50%        9.000      0.719     0.378     0.378             0.672
75%       15.000      0.828     0.571     0.571             0.729
max       64.000      1.000     1.000     1.000             0.965
label threshold 0.327 bad rate 0.401
model_confidence AUROC 0.739
corner_confidence AUROC 0.708
edge_confidence AUROC 0.691
{'hold': 1604, 'tune': 1603} True
coverage_class  dense  sparse
half                         
hold            0.346   0.457
tune            0.346   0.457
is_bad                0      1
model_confidence              
(-0.001, 0.571]   0.283  0.717
(0.571

In [29]:
import json
d=pd.read_csv(f'{OUT}/phase4_labeled.csv')
o=d.set_index('building_id')
order=lambda b:(index_by_id.loc[b].shard,index_by_id.loc[b].row)
RETRY_CFGS={'A':dict(score_thresh=0.45,seed_tag='r1'),
            'B':dict(score_thresh=0.55,seed_tag='r1'),
            'C':dict(score_thresh=0.5,cluster_m=0.35,min_samples=2,seed_tag='r1'),
            'D':dict(score_thresh=0.5,n_passes=16,seed_tag='r1')}
POOL_MAX=0.65

chk=d.building_id.sample(5,random_state=3).astype(int).tolist()
dif=[abs(score_building(load_item(b),BASE)['model_confidence']-o.loc[b,'model_confidence'])+abs(score_building(load_item(b),BASE)['min_f1']-o.loc[b,'min_f1']) for b in chk]
print('determinism vs stored scores, max diff:',max(dif))

def summ(g):
    dF=g.new_f1-g.orig_f1; kept=np.where(g.new_mc>g.orig_mc,g.new_f1,g.orig_f1)
    return pd.Series(dict(n=len(g),improved=(dF>0.02).mean(),same=(dF.abs()<=0.02).mean(),worse=(dF<-0.02).mean(),
        share_kept=(g.new_mc>g.orig_mc).mean(),kept_gain=(kept-g.orig_f1).mean()))

bp=f'{OUT}/retry_outputs/retry_best.json'
if os.path.exists(bp): BEST=json.load(open(bp))['best']
else:
    pool=d[(d.half=='tune')&(d.model_confidence<POOL_MAX)]
    ids=sorted(pool.building_id.sample(min(120,len(pool)),random_state=42).astype(int).tolist(),key=order)
    sel=[]
    for b in ids:
        it=load_item(b)
        for k,c in RETRY_CFGS.items():
            n=score_building(it,{**BASE,**c})
            sel.append(dict(building_id=b,cfg=k,orig_mc=o.loc[b,'model_confidence'],new_mc=n['model_confidence'],orig_f1=o.loc[b,'min_f1'],new_f1=n['min_f1']))
    sel=pd.DataFrame(sel); sel.to_csv(f'{OUT}/retry_outputs/retry_config_selection.csv',index=False)
    tab=pd.DataFrame({k:summ(g) for k,g in sel.groupby('cfg')}).T
    print(tab.round(3))
    BEST=tab.kept_gain.astype(float).idxmax(); json.dump({'best':BEST,'cfgs':RETRY_CFGS},open(bp,'w'))
print('BEST retry config',BEST,RETRY_CFGS[BEST])

rp=f'{OUT}/retry_outputs/retry_results.csv'
rows=pd.read_csv(rp).to_dict('records') if os.path.exists(rp) else []
seen={int(r['building_id']) for r in rows}
ids_all=sorted(d[d.model_confidence<POOL_MAX].building_id.astype(int).tolist(),key=order)
todo=[b for b in ids_all if b not in seen]
print('retry pool',len(ids_all),'| done',len(seen),'| to do',len(todo))
t0=time.time()
for i,b in enumerate(todo):
    n=score_building(load_item(b),{**BASE,**RETRY_CFGS[BEST]})
    rows.append(dict(building_id=b,cfg=BEST,orig_mc=o.loc[b,'model_confidence'],new_mc=n['model_confidence'],
        orig_f1=o.loc[b,'min_f1'],new_f1=n['min_f1'],orig_corners=o.loc[b,'n_corners'],new_corners=n['n_corners']))
    if (i+1)%100==0:
        pd.DataFrame(rows).to_csv(rp,index=False); print(len(rows),'of',len(ids_all),round((time.time()-t0)/60,1),'min')
r=pd.DataFrame(rows); r.to_csv(rp,index=False)
r=r.merge(d[['building_id','half']],on='building_id')
print({h:summ(g).round(3).to_dict() for h,g in r.groupby('half')})
print('done',len(r))

determinism vs stored scores, max diff: 5.551115123125783e-17
       n  improved   same  worse  share_kept  kept_gain
A  120.0     0.358  0.150  0.492       0.567      0.010
B  120.0     0.350  0.142  0.508       0.750      0.006
C  120.0     0.333  0.142  0.525       0.575     -0.007
D  120.0     0.358  0.225  0.417       0.567      0.023
BEST retry config D {'score_thresh': 0.5, 'n_passes': 16, 'seed_tag': 'r1'}
retry pool 1239 | done 0 | to do 1239
100 of 1239 2.3 min
200 of 1239 4.7 min
300 of 1239 7.0 min
400 of 1239 9.4 min
500 of 1239 11.8 min
600 of 1239 14.1 min
700 of 1239 16.5 min
800 of 1239 18.9 min
900 of 1239 21.2 min
1000 of 1239 23.6 min
1100 of 1239 25.9 min
1200 of 1239 28.3 min
{'hold': {'n': 641.0, 'improved': 0.441, 'same': 0.192, 'worse': 0.367, 'share_kept': 0.507, 'kept_gain': 0.03}, 'tune': {'n': 598.0, 'improved': 0.418, 'same': 0.231, 'worse': 0.351, 'share_kept': 0.513, 'kept_gain': 0.026}}
done 1239


In [30]:
import os, json, itertools, numpy as np, pandas as pd
OUT='/kaggle/working/phase4_decision'
d=pd.read_csv(f'{OUT}/phase4_labeled.csv')
r=pd.read_csv(f'{OUT}/retry_outputs/retry_results.csv')
thr=float(d.loc[d.is_bad==1,'min_f1'].max())
T=d.merge(r[['building_id','new_mc','new_f1']],on='building_id',how='left')
assert T.loc[T.model_confidence<0.65,'new_mc'].notna().all()
kn=(T.new_mc>T.model_confidence).values
T['final_mc']=np.where(kn,T.new_mc,T.model_confidence)
T['final_f1']=np.where(kn,T.new_f1,T.min_f1)
P=dict(L=2.0,c_rev=1.0,c_retry=0.2)

def simulate(T,c):
    mc=T.model_confidence.values; fmc=T.final_mc.values; ff=T.final_f1.values; f0=T.min_f1.values
    n=len(T); mid_final='ACCEPT_FLAG' if c['mid']=='ACCEPT_FLAG' else 'ESCALATE'
    hi=mc>=c['t_acc']; lo=mc<c['t_low']; ri=lo&(c['max_retry']>=1)
    ph=fmc>=c['t_acc']; pl=fmc<c['t_low']
    first=np.full(n,mid_final,dtype=object); final=np.full(n,mid_final,dtype=object); reason=np.full(n,'MID_CONF',dtype=object)
    first[hi]='ACCEPT'; final[hi]='ACCEPT'; reason[hi]='HIGH_CONF'
    m=lo&~ri; first[m]='ESCALATE'; final[m]='ESCALATE'; reason[m]='LOW_MODEL_CONF'
    first[ri]='RETRY'
    m=ri&ph; final[m]='ACCEPT'; reason[m]='RETRY_FIXED'
    m=ri&pl; final[m]='ESCALATE'; reason[m]='RETRY_FAILED'
    m=ri&~ph&~pl; final[m]=mid_final; reason[m]='RETRY_MID'
    return dict(first=first,final=final,reason=reason,retried=ri,eff_f1=np.where(ri,ff,f0),eff_mc=np.where(ri,fmc,mc))

def summarize(S,y,idx=None):
    sl=slice(None) if idx is None else idx
    fin=S['final'][sl]; ri=S['retried'][sl]; ef=S['eff_f1'][sl]; yy=y[sl]
    acc=np.isin(fin,['ACCEPT','ACCEPT_FLAG']); esc=fin=='ESCALATE'; eb=ef<=thr
    cost=np.where(acc,P['L']*eb,0.0)+np.where(esc,P['c_rev'],0.0)+np.where(ri,P['c_retry'],0.0)
    caught=esc|(ri&acc&~eb); bad=yy==1
    return dict(cost=cost.mean(),review_rate=esc.mean(),accepted_share=acc.mean(),
        accepted_bad_rate=eb[acc].mean() if acc.any() else np.nan,accepted_mean_f1=ef[acc].mean() if acc.any() else np.nan,
        bad_caught=caught[bad].mean() if bad.any() else np.nan,good_escalated=esc[~bad].mean() if (~bad).any() else np.nan,retried=ri.mean())

tune=T[T.half=='tune'].reset_index(drop=True); hold=T[T.half=='hold'].reset_index(drop=True)
ytu=tune.is_bad.values; yho=hold.is_bad.values
rows=[]
for ta,tl,mr,mid in itertools.product([0.65,0.69,0.715,0.75,0.80],[0.55,0.60,0.633,0.65],[0,1],['ACCEPT_FLAG','ESCALATE']):
    if ta<=tl: continue
    c=dict(t_acc=ta,t_low=tl,max_retry=mr,mid=mid)
    rows.append({**c,**summarize(simulate(tune,c),ytu)})
G=pd.DataFrame(rows)
bt=[]
for b in [0.1,0.2,0.3,0.4]:
    g=G[G.review_rate<=b]
    if len(g): bt.append({'budget':b,**g.sort_values('cost').iloc[0].to_dict()})
print(pd.DataFrame(bt).round(3).to_string(index=False))
BUDGET=0.30
best=G[G.review_rate<=BUDGET].sort_values('cost').iloc[0]
cfg=dict(t_acc=float(best.t_acc),t_low=float(best.t_low),max_retry=int(best.max_retry),mid=str(best.mid))
print('CHOSEN',cfg,'tune review',round(best.review_rate,3))

def mk(final,retried,eff_f1): return dict(final=np.asarray(final,dtype=object),retried=retried,eff_f1=eff_f1)
n_h=len(hold); f0h=hold.min_f1.values
S_pol=simulate(hold,cfg)
S_all=mk(np.full(n_h,'ACCEPT',dtype=object),np.zeros(n_h,bool),f0h)
tau=float(np.quantile(tune.model_confidence,best.review_rate))
S_conf=mk(np.where(hold.model_confidence.values>=tau,'ACCEPT','ESCALATE'),np.zeros(n_h,bool),f0h)
S_orc=mk(np.where(yho==1,'ESCALATE','ACCEPT'),np.zeros(n_h,bool),f0h)
pols={'accept_all':S_all,'confidence_only':S_conf,'rule_policy':S_pol,'oracle':S_orc}
rg=np.random.default_rng(42); B=[rg.integers(0,n_h,n_h) for _ in range(1000)]
pt={k:summarize(S,yho) for k,S in pols.items()}
bs={k:pd.DataFrame([summarize(S,yho,ix) for ix in B]) for k,S in pols.items()}
tab=[]
for k in pols:
    row={'policy':k}
    for m in ['cost','accepted_bad_rate','review_rate']:
        lo_,hi_=np.nanpercentile(bs[k][m],[2.5,97.5]); row[m]=f'{pt[k][m]:.3f} [{lo_:.3f},{hi_:.3f}]'
    for m in ['accepted_mean_f1','bad_caught','good_escalated']: row[m]=round(pt[k][m],3)
    tab.append(row)
print(pd.DataFrame(tab).to_string(index=False))
for m in ['cost','accepted_bad_rate']:
    dd=bs['rule_policy'][m]-bs['confidence_only'][m]
    print('policy minus confidence-only',m,round(pt['rule_policy'][m]-pt['confidence_only'][m],4),np.nanpercentile(dd,[2.5,97.5]).round(4))
print('oracle best cost:',pt['oracle']['cost']==min(v['cost'] for v in pt.values()),'| accept-all worst:',pt['accept_all']['cost']==max(v['cost'] for v in pt.values()))
cv=hold.coverage_class.values
print(pd.DataFrame({c:summarize(S_pol,yho,np.where(cv==c)[0]) for c in ['dense','sparse']}).T.round(3))
ri=S_pol['retried']
if ri.any():
    dF=hold.final_f1.values[ri]-f0h[ri]; yb=yho[ri]==1
    print('retried on hold',int(ri.sum()),'| improved',round((dF>0.02).mean(),3),'same',round((np.abs(dF)<=0.02).mean(),3),'worse',round((dF<-0.02).mean(),3),
          '| bad->good after retry',round((hold.final_f1.values[ri][yb]>thr).mean(),3))
print('action mix on hold:',pd.Series(S_pol['final']).value_counts(normalize=True).round(3).to_dict())

S=simulate(T,cfg)
dec=pd.DataFrame(dict(building_id=T.building_id,half=T.half,coverage_class=T.coverage_class,action_first=S['first'],action_final=S['final'],reason=S['reason'],
    retry_count=S['retried'].astype(int),model_confidence_before=T.model_confidence,model_confidence_after=S['eff_mc'],
    min_f1_before=T.min_f1,min_f1_after=S['eff_f1'],is_bad_before=T.is_bad,is_bad_after=(S['eff_f1']<=thr).astype(int)))
dec.to_csv(f'{OUT}/phase4_decisions.csv',index=False)
q=dec[dec.action_final=='ESCALATE'].sort_values('model_confidence_after').reset_index(drop=True)
q.insert(0,'priority',np.arange(1,len(q)+1)); q[['priority','building_id','half','coverage_class','reason','model_confidence_after']].to_csv(f'{OUT}/escalation_queue.csv',index=False)
bp=f'{OUT}/retry_outputs/retry_best.json'
json.dump({'policy':cfg,'label_threshold_min_f1':thr,'cost_model':P,'review_budget':BUDGET,'confidence_only_tau':tau,'retry_config':json.load(open(bp)) if os.path.exists(bp) else None},open(f'{OUT}/policy_config.json','w'),indent=2)
json.dump({'heldout_n':int(n_h),'heldout_metrics':{k:{m:float(v) for m,v in pt[k].items()} for k in pt}},open(f'{OUT}/phase4_summary.json','w'),indent=2)
print('queue size',len(q),'| decisions',len(dec))

 budget  t_acc  t_low  max_retry         mid  cost  review_rate  accepted_share  accepted_bad_rate  accepted_mean_f1  bad_caught  good_escalated  retried
    0.1   0.65  0.550          0 ACCEPT_FLAG 0.767        0.070           0.930              0.375             0.450       0.129           0.030      0.0
    0.2   0.65  0.600          0 ACCEPT_FLAG 0.729        0.174           0.826              0.336             0.471       0.307           0.085      0.0
    0.3   0.65  0.633          0 ACCEPT_FLAG 0.709        0.288           0.712              0.296             0.497       0.474           0.163      0.0
    0.4   0.65  0.550          0    ESCALATE 0.707        0.373           0.627              0.267             0.517       0.583           0.233      0.0
CHOSEN {'t_acc': 0.65, 't_low': 0.633, 'max_retry': 0, 'mid': 'ACCEPT_FLAG'} tune review 0.288
         policy                cost   accepted_bad_rate         review_rate  accepted_mean_f1  bad_caught  good_escalated
     accept_a

In [31]:
def simulate2(T,c):
    mc=T.model_confidence.values; fmc=T.final_mc.values; ff=T.final_f1.values; f0=T.min_f1.values
    n=len(T); mid_final='ACCEPT_FLAG' if c['mid']=='ACCEPT_FLAG' else 'ESCALATE'
    hi=mc>=c['t_acc']; ri=mc<c['t_low']; ok=fmc>=c['t_post']
    first=np.full(n,mid_final,dtype=object); final=np.full(n,mid_final,dtype=object); reason=np.full(n,'MID_CONF',dtype=object)
    first[hi]='ACCEPT'; final[hi]='ACCEPT'; reason[hi]='HIGH_CONF'
    first[ri]='RETRY'
    m=ri&ok; final[m]='ACCEPT'; reason[m]='RETRY_FIXED'
    m=ri&~ok; final[m]='ESCALATE'; reason[m]='RETRY_FAILED'
    return dict(first=first,final=final,reason=reason,retried=ri,eff_f1=np.where(ri,ff,f0),eff_mc=np.where(ri,fmc,mc))

rows=[]
for ta,tl,tp,mid in itertools.product([0.65,0.69,0.715,0.75],[0.55,0.60,0.633,0.65],[0.58,0.60,0.62,0.633,0.65,0.69],['ACCEPT_FLAG','ESCALATE']):
    if ta<=tl or tp>ta or tp<tl: continue
    c=dict(t_acc=ta,t_low=tl,t_post=tp,mid=mid)
    rows.append({**c,**summarize(simulate2(tune,c),ytu)})
G2=pd.DataFrame(rows)
b2=G2[G2.review_rate<=BUDGET].sort_values('cost').iloc[0]
cfg2=dict(t_acc=float(b2.t_acc),t_low=float(b2.t_low),t_post=float(b2.t_post),mid=str(b2.mid))
print('tune cost: with retry',round(b2.cost,4),'| no retry',round(best.cost,4))
print('CHOSEN (retry variant)',cfg2,'tune review',round(b2.review_rate,3),'retried',round(b2.retried,3))
S2=simulate2(hold,cfg2)
tau2=float(np.quantile(tune.model_confidence,b2.review_rate))
S_c2=mk(np.where(hold.model_confidence.values>=tau2,'ACCEPT','ESCALATE'),np.zeros(n_h,bool),f0h)
p2={'accept_all':S_all,'confidence_only':S_c2,'rule_no_retry':S_pol,'rule_with_retry':S2,'oracle':S_orc}
pt2={k:summarize(S,yho) for k,S in p2.items()}
bs2={k:pd.DataFrame([summarize(S,yho,ix) for ix in B]) for k,S in p2.items()}
tab=[]
for k in p2:
    row={'policy':k}
    for m in ['cost','accepted_bad_rate','review_rate']:
        lo_,hi_=np.nanpercentile(bs2[k][m],[2.5,97.5]); row[m]=f'{pt2[k][m]:.3f} [{lo_:.3f},{hi_:.3f}]'
    for m in ['accepted_mean_f1','bad_caught','good_escalated','retried']: row[m]=round(pt2[k][m],3)
    tab.append(row)
print(pd.DataFrame(tab).to_string(index=False))
for ref in ['confidence_only','accept_all']:
    for m in ['cost','accepted_bad_rate']:
        dd=bs2['rule_with_retry'][m]-bs2[ref][m]
        print('rule_with_retry minus',ref,m,round(pt2['rule_with_retry'][m]-pt2[ref][m],4),np.nanpercentile(dd,[2.5,97.5]).round(4))
ri=S2['retried']
if ri.any():
    ffh=hold.final_f1.values[ri]; dF=ffh-f0h[ri]; yb=yho[ri]==1; okr=(hold.final_mc.values[ri]>=cfg2['t_post'])
    print('retried on hold',int(ri.sum()),'| improved',round((dF>0.02).mean(),3),'same',round((np.abs(dF)<=0.02).mean(),3),'worse',round((dF<-0.02).mean(),3))
    print('retry success (bad -> good after retry):',round((ffh[yb]>thr).mean(),3),'| accepted after retry:',round(okr.mean(),3),'| bad rate among those accepted:',round((ffh[okr]<=thr).mean(),3) if okr.any() else None)

tune cost: with retry 0.7606 | no retry 0.7093
CHOSEN (retry variant) {'t_acc': 0.65, 't_low': 0.633, 't_post': 0.65, 'mid': 'ACCEPT_FLAG'} tune review 0.249 retried 0.288
         policy                cost   accepted_bad_rate         review_rate  accepted_mean_f1  bad_caught  good_escalated  retried
     accept_all 0.802 [0.754,0.849] 0.401 [0.377,0.425] 0.000 [0.000,0.000]             0.430       0.000           0.000    0.000
confidence_only 0.731 [0.690,0.774] 0.317 [0.290,0.345] 0.265 [0.244,0.287]             0.483       0.418           0.162    0.000
  rule_no_retry 0.723 [0.684,0.762] 0.299 [0.271,0.325] 0.313 [0.290,0.335]             0.494       0.488           0.196    0.000
rule_with_retry 0.776 [0.735,0.817] 0.306 [0.281,0.333] 0.261 [0.240,0.281]             0.490       0.445           0.154    0.313
         oracle 0.401 [0.377,0.425] 0.000 [0.000,0.000] 0.401 [0.377,0.425]             0.580       1.000           0.000    0.000
rule_with_retry minus confidence_only cost

In [32]:
import pandas as pd, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
OUT='/kaggle/working/phase4_decision'
dec=pd.read_csv(f'{OUT}/phase4_decisions.csv'); h=dec[dec.half=='hold']
fig,ax=plt.subplots(1,2,figsize=(11,4))
h.action_final.value_counts(normalize=True).plot.bar(ax=ax[0],color='steelblue'); ax[0].set_title('Held-out: action share'); ax[0].set_ylabel('share of buildings')
h.groupby('action_final').is_bad_after.mean().plot.bar(ax=ax[1],color='indianred'); ax[1].set_title('Held-out: share bad, by action'); ax[1].set_ylabel('share bad')
plt.tight_layout(); plt.savefig(f'{OUT}/phase4_summary.png',dpi=150)